# Lab 06 — Preparing a RealWaste annotation pilot

Prepare a small image-classification pilot from Project 03: **load the manifest → select training images → create local-file URLs → assign workers → export CSV**. Run this notebook from `labs/Lab06`.

## 1. Load and inspect the dataset

The manifest stores each image path relative to the manifest directory. We use only its training split; validation and test images remain reserved for model evaluation.

In [ ]:
from pathlib import Path
from urllib.parse import quote
import os
import sys
from io import BytesIO

import matplotlib.pyplot as plt
import pandas as pd
import requests
from PIL import Image, UnidentifiedImageError
from IPython.display import display

LAB_DIR = Path.cwd().resolve()  # Run from labs/Lab06.
REPO_ROOT = LAB_DIR.parents[1]
sys.path.insert(0, str(LAB_DIR))
from utils.assignment import prepare_image_for_assignment

MANIFEST_PATH = REPO_ROOT / "project_baselines/project_03/data/prepared/realwaste/manifest.csv"
LS_DOCUMENT_ROOT = (REPO_ROOT / "data").resolve()
images = pd.read_csv(MANIFEST_PATH)
train = images[images["split"] == "train"].copy()
print("Training images:", len(train))
display(train["label"].value_counts().rename("training images"))
display(train[["image_id", "image_path", "label", "split"]].head())

In [ ]:
preview = train.sample(min(4, len(train)), random_state=42)
fig, axes = plt.subplots(1, len(preview), figsize=(3.2 * len(preview), 3.5), squeeze=False)
for ax, (_, row) in zip(axes[0], preview.iterrows()):
    with Image.open((MANIFEST_PATH.parent / row["image_path"]).resolve()) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row["label"])
    ax.axis("off")
fig.tight_layout()

## 2. Select pilot examples

This random sample is a starting point. Inspect the candidates and replace the sampling step with your own selection strategy if the pilot should include specific boundary cases.

In [ ]:
SAMPLE_SIZE = 18
RANDOM_SEED = 42

pilot = train.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).copy().reset_index(drop=True)
display(pilot[["image_id", "image_path", "label", "split"]])
display(pilot["label"].value_counts().rename("pilot images"))

## 3. Prepare local-file image URLs

Resolve each path from the manifest directory, verify that the file exists under the Label Studio document root, and encode its relative path. The source images are not copied.

In [ ]:
pilot["file_path"] = pilot["image_path"].map(lambda value: (MANIFEST_PATH.parent / value).resolve())
missing = pilot.loc[~pilot["file_path"].map(Path.is_file), ["image_id", "file_path"]]

if not missing.empty:
    raise FileNotFoundError(f"Selected image files are missing:\n{missing.to_string(index=False)}")
outside = pilot.loc[~pilot["file_path"].map(lambda path: path.is_relative_to(LS_DOCUMENT_ROOT)), ["image_id", "file_path"]]

if not outside.empty:
    raise ValueError(f"Selected images are outside the Label Studio document root:\n{outside.to_string(index=False)}")

pilot["image"] = pilot["file_path"].map(
    lambda path: "/data/local-files/?d=" + quote(path.relative_to(LS_DOCUMENT_ROOT).as_posix(), safe="/")
)
print("All", len(pilot), "selected image files exist under", LS_DOCUMENT_ROOT)
display(pilot[["image_id", "image"]].head())

## 4. Assign images to annotators

The image helper reuses the same seeded worker allocation as the text-classification pilot. The worker names organize the pilot; they do not restrict access.

In [ ]:
NUM_WORKERS = 3
NUM_RATINGS_PER_EXAMPLE = 2

selected, tasks, assignments = prepare_image_for_assignment(
    pilot, id_col="image_id", image_col="image", sample_size=len(pilot),
    num_workers=NUM_WORKERS, num_ratings_per_example=NUM_RATINGS_PER_EXAMPLE,
    seed=RANDOM_SEED,
)
display(tasks.head())
display(assignments.head())
display(assignments["worker_id"].value_counts().rename("assignments"))

## 5. Export the pilot files

Import only the task file into Label Studio. Keep the reference labels separate for later comparison.

In [ ]:
EXPORT_DIR = LAB_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
TASKS_CSV = EXPORT_DIR / "realwaste_tasks.csv"
REFERENCE_CSV = EXPORT_DIR / "realwaste_reference.csv"
ASSIGNMENTS_CSV = EXPORT_DIR / "realwaste_assignments.csv"

reference = selected[["image_id", "label"]].rename(columns={"image_id": "example_id", "label": "reference_label"})
tasks.to_csv(TASKS_CSV, index=False)
reference.to_csv(REFERENCE_CSV, index=False)
assignments.to_csv(ASSIGNMENTS_CSV, index=False)
print("Label Studio tasks:", TASKS_CSV)
print("Private reference:", REFERENCE_CSV)
print("Assignment manifest:", ASSIGNMENTS_CSV)
display(tasks.head())